In [1]:
import json
import pennylane as qp
import pennylane.numpy as np

In [ ]:
n_qubits = 4

dev = qp.device('default.qubit', wires = n_qubits)

@qp.qnode(dev)
def iqp_circuit(weights, pattern, observable):

    """This quantum circuit implements the IQP circuit with the given weight, pattern, and observable
    Args:
        weights (np.array(float)): The parameters of the IQP circuit.
        pattern (float): The pattern of the rotation groups as per challenge description.
        observable (np.array(int)): The array describing the observable as per challenge description

    Returns:
        (numpy.array): The expectation value of the observable for the prepared IQP state.
    """
    
    # Put your code here #
    for i in range(n_qubits):
        qp.Hadamard(wires=i)

    # Return the expectation value
    for j, group in enumerate(pattern):
        for wires in group:
            p_w = len(wires) * 'Z'
            qp.PauliRot(weights[j], pauli_word=p_w, wires=wires)

    for i in range(n_qubits):
        qp.Hadamard(wires=i)

    active_wires = [i for i, w in enumerate(observable) if w == 1]

    if len(active_wires) > 0:
        ob = qp.prod(*[qp.PauliZ(wires=i) for i in active_wires])
    else:
        ob = qp.PauliI(wires=0)


    return qp.expval(ob)

    def optimize_iqp(pattern, observable):

    """This function finds the weights that optimize the IQP circuit built above
    Args:
        pattern (float): The pattern of the rotation groups as per challenge description.
        observable (np.array(int)): The array describing the observable as per challenge description

    Returns:
        (numpy.array): The expectation value of the observable for the prepared IQP state.
    """

    # Initialize the weights
    weights = np.array(np.random.uniform(0, 2*np.pi, len(pattern)), requires_grad=True)

    # Build an optimizer
    opt = qp.AdamOptimizer(stepsize=0.1)

    for i in range(200):
        weights, cost = opt.step_and_cost(iqp_circuit, weights, pattern=pattern, observable=observable)

    # Return the optimizing weights
    return weights